In [4]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
    print(f'Uploaded file: {filename}')

Saving 2022_年間売上表.xlsx to 2022_年間売上表 (1).xlsx
Saving 2023_年間売上表.xlsx to 2023_年間売上表 (1).xlsx
Uploaded file: 2022_年間売上表 (1).xlsx
Uploaded file: 2023_年間売上表 (1).xlsx


In [6]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

# ====== 入力ファイル（Colabにアップロード済の想定） ======
file_2022 = "2022_年間売上表.xlsx"
file_2023 = "2023_年間売上表.xlsx"

# ====== 1) Excel → DataFrame（Sheet1） ======
df_2022 = pd.read_excel(file_2022, sheet_name="Sheet1")
df_2023 = pd.read_excel(file_2023, sheet_name="Sheet1")

# ====== 2) 売上年の付与（売上年列が無いケースに対応） ======
# もし売上年列が元データに存在する場合はそのまま使い、
# 無い場合はファイル名から付与する
if "売上年" not in df_2022.columns:
    df_2022["売上年"] = 2022
if "売上年" not in df_2023.columns:
    df_2023["売上年"] = 2023

# ====== 3) データ連結 ======
df_all = pd.concat([df_2022, df_2023], ignore_index=True)

# ====== 4) 商品×売上年で合計金額に集約 ======
# ※「金額」列名は課題データに合わせて必要なら変更してください（例：売上、合計金額など）
summary = (
    df_all.groupby(["商品", "売上年"], as_index=False)["金額（千円）"]
    .sum()
    .rename(columns={"金額（千円）": "合計金額（千円）"})
)

# ====== 5) 新規Excelに書き込み ======
output_file = "売上集計表.xlsx"
summary.to_excel(output_file, sheet_name="Sheet1", index=False)

# ====== 6) openpyxlでヘッダー（1行目）を薄いグレー(#F2F2F2)にする ======
wb = load_workbook(output_file)
ws = wb["Sheet1"]

fill = PatternFill(patternType="solid", fgColor="F2F2F2")  # ← #F2F2F2

# 1行目の最終列まで塗りつぶし
for cell in ws[1]:
    cell.fill = fill

wb.save(output_file)